**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 11 - Gap Analysis

**This notebook answers one research question.**

**Research Question:** What are the biggest gaps, and how far is each state from the top?

This notebook measures the gaps. It does not explain why the gaps exist or how they should
be reduced. Those questions are covered later.

## Setup

Load the scores and the cluster groups. Work out two benchmarks: the national average
(current position) and the top-group average (improvement target).

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
from src import scoring

reports_dir = root / "version2" / "reports"

scores = pd.read_csv(root / "results" / "indicator_scores.csv", index_col=0)
index = pd.read_csv(root / "results" / "competitiveness_index.csv", index_col=0)
clusters = pd.read_csv(reports_dir / "state_clusters.csv", index_col=0)["cluster"]
ranked = index[index["Rank"].notna()].index

national_avg = scores.loc[ranked].mean()
top_group = clusters[clusters == "Strong on both parts"].index
top_group_avg = scores.loc[top_group].mean()

print("ranked:", len(ranked), "| top-group states:", len(top_group))
print("\nbenchmarks by indicator:")
print(pd.DataFrame({"national_avg": national_avg, "top_group_avg": top_group_avg}).round(3).to_string())

ranked: 33 | top-group states: 12

benchmarks by indicator:
                     national_avg  top_group_avg
ger                         0.472          0.620
life_expectancy             0.573          0.588
unemployment_rate           0.625          0.649
cd_ratio                    0.382          0.458
per_capita_power            0.067          0.115
td_losses                   0.662          0.780
road_density                0.195          0.131
factory_density             0.355          0.650
msme_density                0.451          0.607
manufacturing_share         0.366          0.589
investment_rate             0.312          0.560


## Gap vs the national average

For each state and each indicator, how far it is below the national average. A positive
gap means the state is below the average and has room to catch up. This is the state's
current position.

In [2]:
gap_national = (national_avg - scores.loc[ranked])   # positive = below the national average
gap_national.to_csv(reports_dir / "gap_vs_national.csv", index_label="state")

biggest_weakness = gap_national.idxmax(axis=1)
print("saved:", reports_dir / "gap_vs_national.csv")
print("\nmost common biggest weakness (indicator each state is furthest below the average):")
print(biggest_weakness.value_counts().to_string())

saved: D:\india-state-competitiveness-index\version2\reports\gap_vs_national.csv

most common biggest weakness (indicator each state is furthest below the average):
manufacturing_share    6
life_expectancy        6
unemployment_rate      5
road_density           5
ger                    4
factory_density        3
td_losses              2
cd_ratio               1
investment_rate        1


## Gap vs the top group

For each state and each indicator, how far it is below the top group (the "Strong on both
parts" states). This is an improvement target: how much a state would need to rise to match
the strongest states. A positive gap means below the top group.

In [3]:
gap_top = (top_group_avg - scores.loc[ranked])   # positive = below the top group
gap_top.to_csv(reports_dir / "gap_vs_top_group.csv", index_label="state")

development_gap = gap_top.idxmax(axis=1)
print("saved:", reports_dir / "gap_vs_top_group.csv")
print("\nmost common biggest development gap (indicator each state is furthest below the top group):")
print(development_gap.value_counts().to_string())

saved: D:\india-state-competitiveness-index\version2\reports\gap_vs_top_group.csv

most common biggest development gap (indicator each state is furthest below the top group):
factory_density        11
investment_rate         7
manufacturing_share     6
msme_density            2
life_expectancy         2
unemployment_rate       1
ger                     1
road_density            1
cd_ratio                1
td_losses               1


## Per-state diagnostic

For each state, three things: its biggest weakness (furthest below the national average),
its biggest development gap (furthest below the top group), and its easiest gap (the
smallest gap it needs to close to reach the top group).

In [4]:
def easiest_gap(row):
    below = row[row > 0]
    return below.idxmin() if len(below) else None

diagnostic = pd.DataFrame({
    "rank": index.loc[ranked, "Rank"].astype(int),
    "current_weakness": gap_national.idxmax(axis=1),
    "development_gap": gap_top.idxmax(axis=1),
    "easiest_gap": gap_top.apply(easiest_gap, axis=1),
}).sort_values("rank")
diagnostic.to_csv(reports_dir / "state_gap_diagnostic.csv", index_label="state")

print(diagnostic.head(12).to_string())
print("\nmost common easiest gap (smallest gap below the top group):")
print(diagnostic["easiest_gap"].value_counts().to_string())

                  rank     current_weakness      development_gap        easiest_gap
state                                                                              
Goa                  1    unemployment_rate    unemployment_rate           cd_ratio
Tamil Nadu           2         road_density      investment_rate          td_losses
Gujarat              3                  ger                  ger       road_density
Puducherry           4         road_density         road_density           cd_ratio
Telangana            5  manufacturing_share  manufacturing_share                ger
Andhra Pradesh       6         road_density  manufacturing_share    life_expectancy
Punjab               7         road_density  manufacturing_share       road_density
Maharashtra          8         road_density      factory_density       road_density
Himachal Pradesh     9    unemployment_rate         msme_density   per_capita_power
Haryana             10      life_expectancy      life_expectancy    factory_

## Patterns across states

- Against the national average, states' biggest weaknesses are mostly in basic conditions:
  23 of 33 states, against 10 on industry.
- Against the top group, the gap flips to industry: 26 of 33 states have their biggest
  development gap on factory density, investment, manufacturing or MSMEs.
- Most states are already at or above the top group on unemployment, road density, life
  expectancy and schooling. Fewest reach the top group's level on the industry numbers.
- So the weak spot depends on the benchmark. Against the average, it is often basic
  conditions. Against the strongest states, it is almost always industry.